# implementing a neural network from scratch

this notebook builds a simple 2-layer neural network using only raw pytorch tensors (no nn.Module).
the goal is to understand the core mechanics: forward pass, loss computation, backpropagation, and gradient descent.

we train it on a "toy" regression problem: learning the function y = sin(x) from noisy samples.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

## generate toy data

sample 200 points from y = sin(x) + noise

In [ ]:
N = 200
x = torch.linspace(-3, 3, N).unsqueeze(1)
y = torch.sin(x) + 0.1 * torch.randn(N, 1)

plt.scatter(x.numpy(), y.numpy(), s=10, alpha=0.6)
plt.xlabel('x')
plt.ylabel('y')
plt.title('toy dataset: y = sin(x) + noise')
plt.show()

## initialize weights

In [ ]:
hidden_size = 32

# layer 1: input (1) -> hidden (32)
W1 = (torch.randn(1, hidden_size) * 0.5).requires_grad_(True)
b1 = torch.zeros(hidden_size, requires_grad=True)

# layer 2: hidden (32) -> hidden (32)
W2 = (torch.randn(hidden_size, hidden_size) * 0.5).requires_grad_(True)
b2 = torch.zeros(hidden_size, requires_grad=True)

# output layer: hidden (32) -> output (1)
W3 = (torch.randn(hidden_size, 1) * 0.5).requires_grad_(True)
b3 = torch.zeros(1, requires_grad=True)

print(f'total parameters: {W1.numel() + b1.numel() + W2.numel() + b2.numel() + W3.numel() + b3.numel()}')

## forward pass and loss

In [ ]:
def forward(x):
    z1 = x @ W1 + b1
    a1 = torch.relu(z1)
    z2 = a1 @ W2 + b2
    a2 = torch.relu(z2)
    out = a2 @ W3 + b3
    return out

def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()

## training loop

In [ ]:
lr = 0.01
epochs = 2000
params = [W1, b1, W2, b2, W3, b3]
losses = []

for epoch in range(1, epochs + 1):
    # forward pass
    pred = forward(x)
    loss = mse_loss(pred, y)
    losses.append(loss.item())

    # backward pass (compute gradients)
    loss.backward()

    # gradient descent update (use .data to avoid breaking the computation graph)
    with torch.no_grad():
        for p in params:
            p.data -= lr * p.grad.data

    # zero gradients for next iteration
    for p in params:
        p.grad.data.zero_()

    if epoch % 200 == 0 or epoch == 1:
        print(f'epoch {epoch:>4d}  loss: {loss.item():.6f}')

## results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# loss curve
ax1.plot(losses)
ax1.set_xlabel('epoch')
ax1.set_ylabel('MSE loss')
ax1.set_title('training loss')

# predictions vs actual
with torch.no_grad():
    y_pred = forward(x)

ax2.scatter(x.numpy(), y.numpy(), s=10, alpha=0.4, label='data')
ax2.plot(x.numpy(), y_pred.numpy(), color='red', linewidth=2, label='network')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('learned function vs data')
ax2.legend()

plt.tight_layout()
plt.show()